# Decoder Models

The previous notebook used an **encoder**, a model that reads text and returns vectors. This notebook uses a **decoder**, a model that writes text. Both are built from the same transformer layers. The only difference is which tokens each token is allowed to see.

In a decoder, each token may draw only on itself and the tokens before it. This rule is called the **causal mask**, and it is what allows a decoder to be trained to predict the next word:

<img src="assets/attention_masks.png" alt="Two six-by-six grids for the tokens I sat on the river bank. In the encoder grid every cell is filled: each token attends to every token. In the decoder grid only the cells on and below the diagonal are filled: each token attends to itself and the tokens before it." width="800">

A decoder writes in a loop: it gives every token in its vocabulary a score, one token is picked and appended to the text, and the model runs again on the longer text. Temperature, sampling and prompting, the topics of this notebook, all change how that token is picked or what text the loop starts from.

## Learning Objectives

By the end of this notebook, you should be able to:

- Calculate a model's parameter count from its configuration.
- Generate text from logits using temperature sampling.
- Classify reviews by prompting a chat model.

## Setup

In [ ]:
# standard library
import time
from pathlib import Path

# data stack
import numpy as np
import pandas as pd

# deep learning stack
import torch

# nlp stack
import transformers
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

In [ ]:
# As in the other notebooks. The note about unauthenticated requests is printed below
# Python and appears once on the first download regardless.
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

## 1. Reading a Model You Cannot Run

Phi-3-mini is a small model by current standards and still far too big for a laptop: 3.8 billion parameters, 7.6 GB of weights, and written to be run on a GPU.

You do not need the weights to read the design. `AutoConfig` fetches `config.json` alone (about a kilobyte), and that file is the architecture.

In [ ]:
# Downloads config.json and nothing else: no weights, about a kilobyte.
config = AutoConfig.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

In [ ]:
print(f"{config.num_hidden_layers} layers")
print(f"{config.hidden_size} hidden size, the width of every vector inside")
print(f"{config.num_attention_heads} attention heads per layer")
print(f"{config.intermediate_size} wide in each layer's feed-forward network")
print(f"{config.vocab_size:,} tokens in the vocabulary")
print(f"{config.max_position_embeddings:,} tokens of context")

Those six numbers are enough to count the parameters, which is worth doing once so that "3.8 billion" becomes a number you can derive.

Each layer holds two blocks. **Attention** has four square projections (query, key, value and output), each $h \times h$, where $h$ is the hidden size, giving $4h^2$. The **feed-forward network** has three matrices between width $h$ and the intermediate width $i$, giving $3hi$.

Outside the layers sit two matrices of the same shape, $V \times h$ for a vocabulary of $V$ tokens: the **embedding table** that turns ids into vectors on the way in, and the **output head** that turns vectors back into a score per token on the way out. Some models share one matrix for both (`config.tie_word_embeddings` says which), and this one does not, so both are counted.

In [ ]:
h = config.hidden_size
i = config.intermediate_size
V = config.vocab_size

attention = 4 * h * h  # query, key, value and output projections
feed_forward = 3 * h * i  # gate, up and down projections
per_layer = attention + feed_forward
embedding = V * h  # ids to vectors, on the way in
output_head = V * h  # vectors to a score per token, on the way out

total = config.num_hidden_layers * per_layer + embedding + output_head
print("weights tied?", config.tie_word_embeddings)

In [ ]:
print(f"attention per layer:    {attention / 1e6:7.1f}M")
print(f"feed-forward per layer: {feed_forward / 1e6:7.1f}M")
print(
    f"all {config.num_hidden_layers} layers:          {config.num_hidden_layers * per_layer / 1e9:7.2f}B"
)
print(f"embedding + head:       {(embedding + output_head) / 1e6:7.1f}M")
print(f"estimated total:        {total / 1e9:7.4f}B")

**Result.** 3.8209 billion. The checkpoint actually holds 3,821,079,552 parameters, so six numbers and two multiplications came within 199,680, about five thousandths of one per cent. The remainder is the normalization layers, which are a vector each rather than a matrix.

Two conclusions follow from it, and both matter when choosing a model:

- **The layers dominate.** 3.62B of the 3.82B is the stack; the two $V \times h$ matrices together are 5%. That is the reverse of the previous notebook's BERT, where the embedding table alone was a fifth of the model, and the reason is that a big model is mostly *depth*.
- **Width costs quadratically.** Attention is $4h^2$, so doubling the hidden size quadruples that term. This is why model families grow by adding layers and width together rather than either alone.

The output head is worth remembering, because section 3 is about what comes out of it.

## 2. A Model That Fits

Phi-3's weights would not run at a usable speed here, so the rest of the notebook uses **Qwen2.5-0.5B-Instruct**: the same decoder-only design at about an eighth of the size, roughly 1 GB, and fast enough on a CPU. Switching back to Phi-3 is one line, on a machine with a GPU.

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)
model.eval()  # inference mode, so repeated runs give identical output

print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 3. What Generating Actually Is

Run one forward pass and inspect the output.

In [ ]:
prompt = "The capital of France is"
encoded = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    logits = model(**encoded).logits

print("tokens in:", encoded.input_ids.shape[1])
print("logits out:", tuple(logits.shape))

**Result.** Five tokens in, and a `(1, 5, 151936)` block out: for **every position**, a score for **every one of the 151,936 tokens** in the vocabulary. These raw scores are called **logits**.

Only the last position is useful for writing the next word. The others score what could have followed each earlier token, which is how the model is trained but not what we want now. Turning those scores into probabilities is a `softmax`, which exponentiates each one and divides by the total so they sum to 1.

In [ ]:
next_token_scores = logits[0, -1]  # the last position only
probabilities = next_token_scores.softmax(-1)
best = probabilities.topk(5)

pd.DataFrame(
    {
        "token": [repr(tokenizer.decode(i)) for i in best.indices],
        "probability": [round(float(p), 3) for p in best.values],
    }
)

**Result.** `' Paris'` at 0.295, and note the leading space: it is part of the token, as the first notebook's tokenizer comparison showed.

Picking the highest-scoring token and appending it is one step of generation. Do it again with the longer sequence, and again, until an end-of-text token appears. The only real choice is **how** each token gets picked, which is the next section.

## 4. Temperature

Taking the highest-probability token every time is **greedy** decoding. It is reproducible and it is repetitive: the same prompt always gives the same text, and the model tends to fall into loops.

Sampling instead *draws* from the distribution. **Temperature** controls its shape: every logit is divided by $T$ before the softmax, so

- $T < 1$ sharpens the distribution: the most likely token takes almost all the probability, and the output approaches greedy;
- $T = 1$ leaves the model's own probabilities untouched;
- $T > 1$ flattens it, so unlikely tokens get a real chance and the text loses coherence.

In [ ]:
rows = {}
for T in (0.1, 1.0, 2.0):
    scaled = (next_token_scores / T).softmax(-1)
    rows[f"T={T}"] = [round(float(scaled[i]), 3) for i in best.indices]

pd.DataFrame(rows, index=[repr(tokenizer.decode(i)) for i in best.indices])

**Result.** The same five tokens, three temperatures. At $T=0.1$ `' Paris'` holds 1.000 and the rest round to zero, so sampling here behaves exactly like greedy decoding. At $T=2.0$ it keeps only 0.024, and the 151,931 tokens outside this table share almost all the mass, which is how a high temperature produces nonsense rather than creativity.

$T$ cannot be 0, being a divisor; `do_sample=False` is what people mean by "temperature zero".

Temperature only applies when the token is actually drawn, which `generate` does not do by default here, and the defaults are the model's own, not the library's.

In [ ]:
print(model.generation_config)

**Result.** The checkpoint ships `temperature=0.7`, `top_k=20`, `top_p=0.8` and `repetition_penalty=1.1`. Two consequences worth knowing:

- **`top_k` and `top_p` discard the tail before sampling**, so a high temperature would redistribute weight inside a shortlist rather than across the vocabulary. Both are switched off below to isolate temperature.
- **`repetition_penalty=1.1` applies even to greedy decoding**, so the earlier cells were not quite pure argmax.

In [ ]:
for T in (0.3, 1.5):
    torch.manual_seed(0)  # so this cell reads the same way every run
    with torch.no_grad():
        sampled = model.generate(
            **encoded,
            max_new_tokens=25,
            do_sample=True,
            temperature=T,
            top_k=0,
            top_p=1.0,
            repetition_penalty=1.0,
        )
    print(f"T={T}: {tokenizer.decode(sampled[0, encoded.input_ids.shape[1] :])!r}\n")

**Result.** At 0.3 the continuation is flat and factual; at 1.5 it loses coherence, because tokens with a few thousandths of a chance are now being drawn. Nothing about the model changed, only the distribution it sampled from.

**Your turn:** find the temperature at which this model's output stops being coherent. Run the cell above at several temperatures between 0.5 and 2.5, keeping the seed fixed, and say where you would put the ceiling.

In [ ]:
# TODO: generate at several temperatures and read the results.
# Keep torch.manual_seed(0) inside the loop, and keep top_k=0, top_p=1.0.
# your code:

## 5. The Chat Template

Section 3 fed the model a bare string. That is how a **base** model is used, and Qwen2.5-0.5B-**Instruct** is not one: it was fine-tuned on conversations, and it expects its input marked up into turns. Handing it a bare string works, in the sense that tokens come out, but it is not the input the fine-tuning taught it to answer.

`apply_chat_template` does the marking up. Each model family has its own, stored with the tokenizer, so this is another thing you get by loading the tokenizer the model was trained with.

In [ ]:
messages = [
    {
        "role": "user",
        "content": "In one sentence, what is a transformer in machine learning?",
    }
]
templated = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print(templated)

**Result.** Three things the bare string lacked:

- **turn markers**, `<|im_start|>` and `<|im_end|>`, which are single tokens in this vocabulary rather than punctuation the model has to interpret;
- **a system turn**, inserted by the template although nobody wrote one;
- **a trailing `<|im_start|>assistant`**: that is what `add_generation_prompt=True` adds, and it is the cue that makes the next token the start of a reply rather than more of the question.

In [ ]:
encoded = tokenizer(templated, return_tensors="pt")
with torch.no_grad():
    # greedy: no sampling, so this cell gives the same answer every run
    output = model.generate(**encoded, max_new_tokens=40, do_sample=False)

# everything after the prompt is what the model wrote
print(
    tokenizer.decode(output[0, encoded.input_ids.shape[1] :], skip_special_tokens=True)
)

## 6. Prompting: Asking Without Training

The previous notebook classified these reviews two ways, both needing the encoder run over every text first. A generative model can be asked in English instead. 100 reviews are enough to measure on, since each is a separate generation and that is slow.

In [ ]:
reviews = pd.read_csv(Path("data") / "reviews.csv")
sample = reviews.sample(100, random_state=7)
print(sample["sentiment"].value_counts().to_string())

In [ ]:
INSTRUCTION = (
    "Is the sentiment of this product review positive or negative? "
    "Answer with one word."
)


def classify(text, examples=()):
    """Ask the model to label one review, optionally after some worked examples.

    `examples` are (review, label) pairs, added as completed turns of the conversation,
    which is how a chat model is shown what an answer should look like.
    """
    messages = []
    for example_text, example_label in examples:
        messages.append(
            {"role": "user", "content": f"{INSTRUCTION}\n\nReview: {example_text}"}
        )
        messages.append({"role": "assistant", "content": example_label})
    messages.append({"role": "user", "content": f"{INSTRUCTION}\n\nReview: {text}"})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    encoded = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        # three tokens are enough for one word, and they keep the 100 generations fast
        output = model.generate(
            **encoded,
            max_new_tokens=3,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return (
        tokenizer.decode(
            output[0, encoded.input_ids.shape[1] :], skip_special_tokens=True
        )
        .strip()
        .lower()
    )

In [ ]:
def score(examples=(), rows=None):
    """Accuracy over `rows`, counting an answer as correct if it starts with the label."""
    rows = sample if rows is None else rows
    correct = sum(
        classify(row.text, examples).startswith(row.sentiment)
        for row in rows.itertuples()
    )
    return correct / len(rows)


# About 90 seconds: one generation per review, on a CPU.
zero_shot = score()
print(f"zero-shot: {zero_shot:.3f}")

**Result.** 0.880, from an instruction and nothing else. No labeled training set, no fitting, no encoder run over the corpus: a sentence of English.

Now **few-shot**: show the model some worked examples first. There is a way of doing this that used to be standard and now fails, and it is worth seeing fail.

In [ ]:
# The classic form, written as one block of text for the model to continue.
completion_style = (
    "Review: Broke after two days. Total waste of money.\nSentiment: negative\n\n"
    "Review: Arrived early and works exactly as described. Very pleased.\nSentiment: positive\n\n"
    "Review: {text}\nSentiment:"
)


def classify_completion(text, max_new_tokens=3):
    """Label one review with the completion-style block above, for comparison."""
    messages = [{"role": "user", "content": completion_style.format(text=text)}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    encoded = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return (
        tokenizer.decode(
            output[0, encoded.input_ids.shape[1] :], skip_special_tokens=True
        )
        .strip()
        .lower()
    )

In [ ]:
# A longer budget here, to show what it does when not cut off after three tokens.
for row in sample.head(3).itertuples():
    print(
        f"{classify_completion(row.text, max_new_tokens=8)!r:52s} (should be {row.sentiment!r})"
    )

**Result.** Sometimes a bare label, sometimes a sentence of commentary. Scored over the whole sample it is worse than that looks.

In [ ]:
# Another ~90 seconds, so the failure is measured rather than asserted.
answers = [classify_completion(row.text) for row in sample.itertuples()]
correct = sum(
    a.startswith(row.sentiment) for a, row in zip(answers, sample.itertuples())
)
no_label = sum(not a.startswith(("positive", "negative")) for a in answers)

print(f"completion-style: {correct / len(sample):.3f}")
print(f"{no_label} of {len(sample)} did not answer with a label at all")

**Result.** 0.490, chance level, against 0.880 for the plain instruction. 41 of the 100 never produced a label, and those are automatically wrong.

The reason is section 5. A pattern laid out as text to be continued is what a **base** model responds to; this model was fine-tuned to answer the person in the user turn, so it reads the whole block, examples included, as one question and replies conversationally. **Adding examples the wrong way was worse than adding none.**

On a chat model the examples belong in the conversation, as user turns with the assistant's answers already filled in, which is what `classify` builds.

In [ ]:
shots = [
    ("Broke after two days. Total waste of money.", "negative"),
    ("Arrived early and works exactly as described. Very pleased.", "positive"),
]

# Slower than the zero-shot run: every request now carries the examples too. ~3 minutes.
two_shot = score(shots)
print(f"two-shot: {two_shot:.3f}")

**Result.** 0.890 against 0.880: one review in a hundred, for nearly twice the time, because every request now re-sends the examples.

That is the shape of few-shot on a task like this: the model already knows what positive and negative mean, so examples teach it nothing. Few-shot is worth its cost when the *format* is unusual, when the criteria are yours rather than the world's, or when a label name means something particular to your domain.

### What the three notebooks measured

The encoder route scored on 2,500 reviews and this one on 100, so run the encoder's zero-shot over these same 100 to compare like with like.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

encoder = SentenceTransformer(
    "thenlper/gte-small", model_kwargs={"dtype": torch.float32}
)
descriptions = ["A negative review", "A positive review"]
class_names = np.array(["negative", "positive"])

In [ ]:
start = time.perf_counter()
vectors = encoder.encode(sample.text.tolist())
elapsed = time.perf_counter() - start

nearest = cosine_similarity(vectors, encoder.encode(descriptions)).argmax(axis=1)
print(
    f"embeddings, zero-shot: {(class_names[nearest] == sample.sentiment.values).mean():.3f}"
)
print(
    f"{elapsed / len(sample) * 1000:.0f} ms per review, against roughly 900 for a generation"
)

**Result.** 0.900 for the encoder against 0.880 and 0.890 for the generative model, on identical reviews, and about 18 ms against about 900.

| Method | Labels used | Score | Reviews | Cost per text |
|---|---|---|---|---|
| TF-IDF + logistic regression | 500 | 0.779 | 2,500 | microseconds |
| Frozen embeddings + logistic regression | 500 | **0.938** | 2,500 | ~18 ms |
| Embeddings, zero-shot against label descriptions | none | 0.912 | 2,500 | ~18 ms |
| Embeddings, zero-shot | none | 0.900 | 100 | ~18 ms |
| Generative, zero-shot instruction | none | 0.880 | 100 | ~900 ms |
| Generative, two-shot as chat turns | 2, hand-written | 0.890 | 100 | ~1.6 s |
| Generative, two-shot as a text block | 2, hand-written | 0.490 | 100 | ~900 ms |

**Read the last four rows as a tie.** At 100 reviews one answer is one point and the standard error is about three, so 0.880, 0.890 and 0.900 cannot be told apart; what separates them is **fifty times the cost**. The one row that is clearly different is 0.490, and that is a formatting mistake rather than a method.

The clear result is the second row: a frozen encoder with 500 labels beats everything else, at a fiftieth of the cost. What the generative model offers instead is that it can be redirected by rewriting a sentence, and would still work if the task stopped being classification at all.

Two things change that arithmetic: a model a hundred times this size answers far better, and reaching one through an API removes the limit of local hardware. What does not change is that an encoder is very hard to beat on cost for a task that never varies, which is worth remembering before choosing the expensive option.

**Your turn:** the instruction is a free choice, as the label descriptions were in the previous notebook. Rewrite `INSTRUCTION`: be more specific about what counts as positive, ask for lowercase, name the domain, and re-score. Use `sample.head(25)` while experimenting, or each attempt costs a couple of minutes.

In [ ]:
# TODO: write at least two alternative instructions and score each on a smaller sample.
# `score` reads the global INSTRUCTION through `classify`, so reassign it between runs,
# and pass `rows=quick` so each attempt takes seconds rather than minutes.
quick = sample.head(25)
# your code:

## Summary

- A decoder is the encoder's transformer block, stacked the same way. One forward pass returns a **logit** for every token in the vocabulary at every position; generation takes the last position, picks a token, appends it, and repeats.
- A model's `config.json` is about a kilobyte and holds the architecture, so a model far too big to run can still be read and counted: six numbers reproduced Phi-3's 3.82 billion parameters to five thousandths of a per cent. In a large decoder the **layers dominate**, where a small encoder is mostly its embedding table.
- **Temperature** divides the logits before the softmax: below 1 it sharpens towards greedy, above 1 it flattens towards noise. The useful range is roughly 0 to 1.2.
- An instruction-tuned model expects marked-up **turns**, which `apply_chat_template` supplies; the markup differs per model family.
- **Zero-shot prompting** classified reviews at 0.880 with no labels at all. **Few-shot** must be given as completed conversation turns: the old completion-style block scored 0.490 on this model, worse than no examples at all. Given correctly it added one review in a hundred, because the task was never the problem.
- On this fixed classification task the generative model matched the encoder's zero-shot and lost to its trained classifier, at about fifty times the cost per review. Generative models are the better choice where the task is open-ended, not where a classifier already fits.

## Sources

- The architecture walk-through and the logits-to-sampling sequence follow chapter 3 of *Hands-On Large Language Models* by Jay Alammar and Maarten Grootendorst (O'Reilly, 2024), *The Inputs and Outputs of a Trained Transformer LLM* and *Choosing a Single Token from the Probability Distribution*. The comparison of classifiers in section 6 closes chapter 4's own comparison. The book's [companion code](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) is Apache-2.0.
- The book runs `microsoft/Phi-3-mini-4k-instruct` on a GPU. This notebook reads its configuration and generates with [`Qwen/Qwen2.5-0.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), Apache-2.0, which runs on a CPU.
- The reviews are the Amazon Polarity sample of the previous notebook; `data/README.md` has the citation.

The repository [README](README.md) lists every source and license.